# E2 · Table maintenance: snapshots, time travel, compaction, expiry

A sensor job writes a small batch of temperature readings into a table every 5 minutes.
After an hour you will have the three classic problems of a real table:

* a **mistake** to undo (someone deletes good rows),
* **too many small files** (every batch wrote its own file),
* **old snapshots** that keep deleted data around forever.

Iceberg solves each one with a snapshot operation. Keep `README.md` open next to this
notebook: it explains why each step works. Run the cells top to bottom with **Shift+Enter**;
cells marked **YOUR TURN** need a small change first.

## Step 1 · Connect and create the table

In [ ]:
import sys

sys.path.insert(0, "../_shared")
from lakehouse import spark
from trackcheck import namespace

s = spark("E2 table maintenance")
ns = namespace()
table = f"lakehouse.{ns}.readings"

s.sql(f"CREATE NAMESPACE IF NOT EXISTS lakehouse.{ns}")
s.sql(f"""
CREATE TABLE IF NOT EXISTS {table} (
    batch_no      INT,
    device_id     STRING,
    reading_at    TIMESTAMP_NTZ,
    temperature_c DOUBLE)
USING iceberg""")
print("ready:", table)

## Step 2 · One hour of small batches

Each `INSERT` is one commit, so it creates one **snapshot** and (here) one small data file.
Twelve batches of 100 readings = one hour of the sensor job.

In [ ]:
def write_batch(b):
    """Batch b (1-12): 100 readings of 10 devices, 5 minutes after batch b-1."""
    s.sql(f"""
        INSERT INTO {table}
        SELECT {b},
               format_string('dev-%02d', CAST(id % 10 AS INT)),
               TIMESTAMP_NTZ '2026-03-01 08:00:00' + make_dt_interval(0, 0, {b} * 5, id),
               round(18 + (id * 7 + {b} * 13) % 17 + (id % 3) * 0.5, 1)
        FROM range(0, 100, 1, 1)""")


for b in range(1, 13):
    write_batch(b)
print(s.table(table).count(), "rows")

## Step 3 · Look at the history

`.snapshots` lists every commit; `.files` lists the data files the *current* snapshot uses.

In [ ]:
snaps = s.sql(f"SELECT committed_at, snapshot_id, operation FROM {table}.snapshots ORDER BY committed_at")
snaps.show(truncate=False)
first_snapshot = snaps.first()["snapshot_id"]
print("first snapshot:", first_snapshot)
print("data files now:", s.sql(f"SELECT count(*) FROM {table}.files").first()[0])

Twelve files of about one kilobyte each. Every query has to open all of them; with a real
sensor job (one file every few seconds, for months) this "small files problem" makes queries
slow. We fix it in step 7.

## Step 4 · Time travel (YOUR TURN)

A snapshot never changes, so you can query the table **as it was** at any snapshot that
still exists: `VERSION AS OF <snapshot id>` (or `TIMESTAMP AS OF '<time>'`).

In [ ]:
s.sql(f"SELECT count(*) AS rows_then FROM {table} VERSION AS OF {first_snapshot}").show()

Your team wants to keep the very first batch as a reference, even after old snapshots are
cleaned up (step 8). A snapshot is not a backup, so copy the data into a table of its own.

**YOUR TURN:** finish the statement: select everything from `{table}` **as of the first
snapshot**. You should get 100 rows.

In [ ]:
s.sql(f"""
CREATE TABLE IF NOT EXISTS lakehouse.{ns}.readings_first_batch USING iceberg AS
SELECT * FROM {table}   -- TODO: read the table as of the first snapshot
""")
print(s.table(f"lakehouse.{ns}.readings_first_batch").count(), "rows (want 100)")

## Step 5 · An accident

A colleague's cleanup job believes that readings above 30 °C are sensor errors, and deletes
them. They were real. Run the cell to make the same mistake:

In [ ]:
before_delete = s.sql(f"SELECT snapshot_id FROM {table}.snapshots ORDER BY committed_at DESC LIMIT 1").first()[0]
s.sql(f"DELETE FROM {table} WHERE temperature_c > 30")
print("rows now:", s.table(table).count(), "| snapshot before the delete:", before_delete)

## Step 6 · Roll back (YOUR TURN)

Nothing is lost yet: the snapshot before the delete still exists. A **rollback** makes that
snapshot current again. It writes no data; it only moves the table's "current" pointer.

Iceberg's maintenance operations are Spark **procedures**, run with `CALL`. They take the
table name *without* the catalog: `'<namespace>.<table>'`.

**YOUR TURN:** replace `TODO_SNAPSHOT_ID` with the snapshot before the delete (hint: the
variable `before_delete` from step 5 holds it; in an f-string write `{before_delete}`).

In [ ]:
s.sql(f"CALL lakehouse.system.rollback_to_snapshot('{ns}.readings', TODO_SNAPSHOT_ID)").show()
print("rows now:", s.table(table).count(), "(want 1200)")

## Step 7 · Compact the small files

`rewrite_data_files` reads the small files and writes the same rows into fewer, bigger files,
in a new snapshot whose operation is `replace`. Queries see the same data before and after.

In [ ]:
s.sql(f"CALL lakehouse.system.rewrite_data_files(table => '{ns}.readings')").show(truncate=False)
print("data files now:", s.sql(f"SELECT count(*) FROM {table}.files").first()[0])

## Step 8 · Expire old snapshots

Every snapshot keeps its files alive, including the rows deleted in step 5 and the twelve
small files replaced in step 7. `expire_snapshots` removes snapshots older than a point in
time, always keeping the newest `retain_last`, and deletes the files nobody needs any more.
A real job keeps a few days of history; here we keep only the current snapshot.

In [ ]:
s.sql(f"""CALL lakehouse.system.expire_snapshots(
            table => '{ns}.readings', older_than => current_timestamp(), retain_last => 1)""").show(truncate=False)
s.sql(f"SELECT committed_at, snapshot_id, operation FROM {table}.snapshots").show(truncate=False)

Time travel only reaches snapshots that still exist. Try the first snapshot again: it is
gone. That is why you copied the first batch in step 4.

In [ ]:
try:
    s.sql(f"SELECT count(*) FROM {table} VERSION AS OF {first_snapshot}").show()
except Exception as e:
    print("as expected:", str(e).splitlines()[0][:150])
print("readings_first_batch still has", s.table(f"lakehouse.{ns}.readings_first_batch").count(), "rows")

## Step 9 · Check your work

The checkpoint looks at your tables and their snapshots, as you. In a terminal:
`lab-tracks check E2`.

In [ ]:
!python checkpoint.py

In [ ]:
s.stop()